# 02 · Navigation

Notebook `01` ended by saying what was missing:

> Tracking yields carrier phase and code phase per satellite. A position fix additionally needs the navigation message decoded, ephemerides evaluated, and the pseudoranges solved — none of which this notebook does.

This notebook does those three things, and then the two answers they buy: **where the antenna is**, and **what the receiver's clock is doing**.

## What a fix needs that tracking does not give you

Tracking measures one quantity that matters here: `code_phase_ms`, the cumulative code phase. It advances at the *satellite's* code rate, so a difference in code phase is a difference in satellite time — exactly, not approximately. What it does not have is an origin. Acquisition seeded it arbitrarily, so it says *when, relative to some unknown instant*, not *when*.

Three things close that gap.

| Missing | Where it comes from | Cost of getting it wrong |
|---|---|---|
| **Absolute time** | the navigation message: a decoded time of week pins the code phase origin | one millisecond is 300 km |
| **Satellite positions** | broadcast ephemeris, evaluated at the time of *transmission* | metres to tens of metres |
| **A common instant** | interpolating every channel onto one receiver-clock grid | each channel's epochs land on its own code-phase grid, so no two are simultaneous |

With those, a pseudorange is `c · (t_receive − t_transmit)` and four of them give position and clock.

## The receiver's clock is the sample counter

This is worth being explicit about, because it is where the "clock solution" comes from. The receiver here has no clock other than `uptime_ms` — the count of samples that have gone past. That count is not GPS time and never becomes it. Its constant offset is absorbed by the clock bias the position solve estimates, and its *rate error* — the front end's oscillator running fast or slow — shows up as a slope in that estimate.

So the clock bias trace is not a nuisance parameter to be discarded. It is a measurement of the hardware, in parts per million.

## This notebook runs on L5

Both collects shipped with this repository are L5-only, so that is what gets decoded and solved here. The decoders themselves cover all four GPS civil signals — LNAV on L1 C/A, CNAV on L2C and L5, CNAV-2 on L1C — and are exercised against synthetic signals in `tests/`, the same way L1C's tracking is.

## What the shipped collect can and cannot show

Worth knowing before you run it, because it is a property of the capture rather
than of the code.

L5 is carried only by Block IIF and Block III satellites, not by the older IIR and
IIR-M. At this collect's epoch ten GPS satellites were above the horizon and
**exactly four of them transmit L5** — and acquisition finds all four, so nothing
is being missed. But one of those four, G08, sits at 0.1° elevation: about 24
dB-Hz, 25 dB below the others, and it never reaches phase lock. Three satellites
survive to the measurement stage.

Three is enough for everything here except a position fix, which needs four. So
section 8 reports that and steps aside, and section 9 holds the position and
solves the clock — which needs only one satellite, and is how a timing receiver at
a surveyed site runs anyway. Point this notebook at a collect with four locked
satellites and section 8 runs as written.

## Sequence

3. Load notebook `01`'s tracking results
4. Decode the navigation message → absolute time
5. Ephemerides, and the one the satellites themselves sent
6. Skyplot
7. Pseudoranges, and what each correction is worth
8. Position solution
9. Clock solution
10. Broadcast versus precise orbits

## 1. Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import utils
from utils import broadcast_ephemeris, navigation, observables, tracking_io
from utils.nav import cnav
from utils.nav import symbols as nav_symbols
from utils.signal_interfaces import GpsL5

utils.plotting.setup_default_plotting()

## 2. Configuration

Everything adjustable lives here.

### The reference position

The skyplot needs somewhere to stand, and the atmospheric corrections need an
approximate position to compute an elevation from. Both are entirely happy with a
value good to a kilometre.

What the value below is **not** good enough for is a statement about accuracy. The
fix is compared against it in section 6, and until a surveyed position replaces it
that comparison says "the chain is working", not "the receiver is accurate to N
metres".

In [ ]:
# ----- Which tracking run to read -----
# Must match what notebook 01 was run with.
SIGNAL_ID = GpsL5
EXPERIMENT_INDEX = 1

# ----- Reference site -----
# TODO: replace with the surveyed AERO rooftop antenna position.
# Coarse CU Boulder value, good to a few hundred metres.  Fine as an a-priori and
# for the skyplot; NOT a truth reference for accuracy claims.
RECEIVER_REFERENCE_GEO = (40.0076, -105.2659, 1655.0)  # latitude deg, longitude deg, height m

# ----- Measurement grid -----
# One fix every this many milliseconds.  100 ms over a 60 s run is ~600 fixes:
# enough to see the clock drift as a line, few enough to plot honestly.
EPOCH_INTERVAL_MS = 100.0

# ----- Corrections -----
# Turned on one at a time in section 5 to show what each is worth.
CORRECTIONS = navigation.CorrectionSettings(
    satellite_clock=True,
    group_delay=True,
    sagnac=True,
    troposphere=True,
    ionosphere=True,
)

# ----- Orbits -----
# Broadcast ephemeris drives the fix; SP3 is downloaded as an independent check.
USE_PRECISE_ORBIT_CHECK = True

# How much tracking notebook 02 needs.  An L5 CNAV message is 6 s, and finding one
# plus confirming against the next takes ~15 s once the loops have pulled in.
MINIMUM_TRACK_DURATION_MS = 15_000

reference_ecef = navigation.geodetic_to_ecef(*RECEIVER_REFERENCE_GEO)
print(f"Signal:    {SIGNAL_ID.signal_type_id}")
print(f"Reference: {RECEIVER_REFERENCE_GEO[0]:.4f} deg N, "
      f"{RECEIVER_REFERENCE_GEO[1]:.4f} deg E, {RECEIVER_REFERENCE_GEO[2]:.0f} m")
print(f"           ECEF {np.array2string(reference_ecef, precision=1)}")

## 3. Load the tracking results

Notebook `01`'s final cell wrote these. The load is guarded rather than trusting:
a file from a different collect, or one too short to hold a navigation message,
would still produce a position — just the wrong one, with nothing on the plots to
say so.

In [ ]:
collects_dir = utils.environment_variables.get_collects_path()
experiment_name = utils.collect_metadata_utils.list_experiment_names(collects_dir)[
    EXPERIMENT_INDEX
]
metadata = utils.collect_metadata_utils.load_experiment_metadata_from_file(
    collects_dir / experiment_name / "metadata.yml"
)
target_band = SIGNAL_ID.link.id.value
collect_id = next(
    cid for cid in metadata.collect_ids
    if target_band in metadata.channel_configurations[
        metadata.collects[cid].channel_config_id
    ].band_ids
)

tracking_path = (
    utils.environment_variables.get_outputs_path()
    / "tracking"
    / f"{collect_id}_{SIGNAL_ID.signal_type_id}.h5"
)
run = tracking_io.require_tracking_run(
    tracking_path,
    collect_id=collect_id,
    signal_type_id=SIGNAL_ID.signal_type_id,
    minimum_duration_ms=MINIMUM_TRACK_DURATION_MS,
    minimum_signals=4,
)

print(f"Loaded {tracking_path.name}")
print(f"  experiment  {run.experiment_name}")
print(f"  collect     {run.collect_id}")
print(f"  written by  commit {run.git_commit}")
print(f"  satellites  {', '.join(run.signal_ids)}")
for sid in run.signal_ids:
    print(f"    {sid}: {run[sid].duration_ms / 1000:6.1f} s, "
          f"{run[sid].outputs.output_index:,} epochs")

# How many of these are *usable* is not known yet -- a channel that never reached
# phase lock yields no symbols, and one without a decoded time of week has no
# absolute time. Section 4 settles it.
print(f"\n{len(run)} channel(s) tracked. Sections 4 and 5 decide how many carry a")
print("usable measurement; a position fix needs four of them.")

## 4. Decode the navigation message

This is the step that turns tracking into navigation.

### From epochs to symbols

A tracking channel produces one prompt correlator value per *epoch*, and an epoch
is not a symbol. For L5 the two line up once the overlay is stripped: NH20 sync
lets the channel extend its coherent accumulation to 10 ms, which is exactly one
CNAV symbol on L5I, and anchored to the symbol boundary. That alignment is not a
coincidence — it is why `TRACKING_POLICIES["GPS_L5"].synced_coherent_duration_ms`
is 10 and not NH20's more tempting 20.

Two details `utils.nav.symbols` handles that are easy to get wrong:

- **The data is on the imaginary axis.** L5's carrier loop runs on the Q pilot, and
  I and Q are separated by carrier phase, so the data component arrives 90° from
  the carrier reference. Take the real part and you get the noise axis.
- **Epochs before sync are not symbols.** Until NH is wiped off, the overlay is
  still flipping the data component's sign every millisecond. Summing those
  cancels rather than integrating, so they are dropped.

### From symbols to time

CNAV is 300-bit messages, rate-½ convolutionally encoded (K=7, G1=171, G2=133
octal) and protected by CRC-24Q. Every message carries a preamble, the transmitting
PRN, a message type, and a 17-bit time of week in its first 38 bits — so a single
valid CRC anywhere in the stream establishes absolute time.

The decoder searches four interpretations of what tracking hands it, because
tracking resolves neither of two ambiguities: which symbol is a G1 (the encoder's
pair boundary is invisible to a code-tracking loop), and whether the data arrived
inverted (Costas lock is invariant to a 180° flip). The CRC settles both.

One subtlety in the time field: the count times six is SV time at the start of the
**next** message, not this one.

In [ ]:
streams, anchors, decodes = {}, {}, {}

for sid in run.signal_ids:
    stream = nav_symbols.extract(run[sid].outputs, SIGNAL_ID.signal_type_id)
    streams[sid] = stream
    if not len(stream):
        print(f"{sid}: no usable symbols (channel never synced its overlay)")
        continue

    result = cnav.decode(
        stream.soft,
        message_duration_s=cnav.MESSAGE_DURATION_S[SIGNAL_ID.signal_type_id],
        expected_prn=int(sid[1:]),
    )
    decodes[sid] = result
    if not result.synced:
        print(f"{sid}: {len(stream):5,} symbols, no message survived the CRC")
        continue

    anchors[sid] = observables.anchor_from_cnav(stream, result.messages[0], sat_id=sid)
    types = sorted({m.message_type for m in result.messages})
    print(
        f"{sid}: {len(stream):5,} symbols "
        f"(data/quadrature energy {stream.quadrature_energy_ratio:5.1f}), "
        f"{len(result.messages)} message(s), types {types}, "
        f"{'inverted' if result.inverted else 'upright'} "
        f"phase {result.symbol_phase}"
    )
    print(f"      TOW at first message start: {anchors[sid].tow_s:,.1f} s  "
          f"({anchors[sid].source})")

if not anchors:
    raise RuntimeError(
        "No satellite yielded a decoded time of week, so no pseudorange can be "
        "formed. Check that notebook 01 tracked long enough (an L5 CNAV message "
        "is 6 s) and that its channels reached PLL lock."
    )

### Is the decoded time believable?

Three checks, none of which depends on the others.

1. **Every satellite should agree.** They are all transmitting the same system time,
   so their decoded times of week must match to within the spread of their
   transit times — 67 to 86 ms, so a spread of about 20 ms.
2. **Successive messages should step by one message duration.** A CRC that passed
   on a mis-aligned window would still give a plausible single time.
3. **It should agree with the file's own timestamp.** The collect is named
   `20230417_103222`; that is wall-clock, so the comparison is coarse, but being
   out by hours would show.

In [ ]:
tows = {sid: a.tow_s for sid, a in anchors.items()}
spread_ms = (max(tows.values()) - min(tows.values())) * 1e3
print(f"Decoded TOW spread across satellites: {spread_ms:.1f} ms")
print("  (transit times run 67-86 ms, so up to ~20 ms of spread is expected)")

for sid, result in decodes.items():
    if result.synced:
        ok = "consistent" if result.tow_is_consistent() else "INCONSISTENT"
        print(f"  {sid}: {len(result.messages)} messages, TOW sequence {ok}")

# Absolute time, and a cross-check against the code phase acquisition recovered.
# L5 folds NH20 into the acquisition replica, so that phase is already absolute
# modulo 20 ms -- an independent handle on the same instant.
reference_tow = float(np.median(list(tows.values())))
day_seconds = reference_tow % 86400.0
print(f"\nMedian decoded TOW: {reference_tow:,.1f} s into the GPS week")
print(f"  = {int(day_seconds // 3600):02d}:{int(day_seconds % 3600 // 60):02d}"
      f":{day_seconds % 60:05.2f} UTC-ish on that day of week")
print(f"  collect filename says 10:32:22 local")

## 5. Ephemerides

Two sources, for two different purposes.

**Broadcast** (`brdc`, from CDDIS) is what drives the fix. It is what a real
receiver has, it carries the satellite clock polynomial the pseudorange correction
needs, and its errors are part of what a single-frequency fix actually experiences.

**Precise** (IGS SP3) is the check. Comparing the two shows what the broadcast
orbit costs, which is the honest way to say how much of the position error is the
receiver's fault and how much arrived with the signal.

There is also a third source available here and worth noticing: the ephemeris
**decoded from the CNAV messages above**. CNAV parameterises the orbit differently
from LNAV — a semi-major axis stated as a difference from a reference plus a rate,
a mean motion correction with its own rate — so it is not directly comparable
parameter by parameter, but it predicts a position that is.

> The first run downloads from CDDIS using the Earthdata credentials in `.env`, so
> it needs network access and takes a minute.

In [ ]:
from datetime import datetime, timedelta

# The date comes from the collect id; the time of week comes from the signal
# itself, decoded above.  Deriving both from the filename would be circular --
# checking the decoded time against the filename is the whole point of section 4.
collect_date = datetime.strptime(collect_id.split("_")[0], "%Y%m%d")
week, _ = broadcast_ephemeris.gps_week_and_tow(collect_date)
epoch_datetime = broadcast_ephemeris.datetime_from_gps(week, reference_tow)
print(f"Collect epoch: {epoch_datetime}   (GPS week {week}, TOW {reference_tow:,.1f} s)")

# `utils.broadcast_ephemeris` rather than `gnss_tools.misc.rinex_gps_ephemeris`:
# that module cannot be imported (it references a `gnss_tools.rinex` package that
# is actually called `rinex_io`), and its PVT helper calls its own functions with
# the wrong signatures.  Both are bugs in the sibling repository.
records = broadcast_ephemeris.load_brdc_span(
    epoch_datetime - timedelta(hours=2), epoch_datetime + timedelta(hours=2)
)
print(f"Broadcast file holds ephemerides for {len(records)} satellites")

broadcast = broadcast_ephemeris.ephemerides_for(
    sorted(anchors), reference_tow, week, epoch_datetime, records=records
)
for sid, eph in sorted(broadcast.items()):
    age_min = (reference_tow - eph.toe) / 60.0
    print(f"  {sid}: toe {eph.toe:>9,.0f} ({age_min:+6.1f} min)  "
          f"sqrt(A) {eph.sqrt_a:.3f}  e {eph.e:.6f}  af0 {eph.af0 * 1e6:+7.3f} us")

missing = sorted(set(anchors) - set(broadcast))
if missing:
    print(f"\nDropping {', '.join(missing)}: no usable ephemeris, so no position for them.")
    for sid in missing:
        anchors.pop(sid)
if not anchors:
    raise RuntimeError(
        "No satellite has both a decoded time of week and a usable ephemeris, so "
        "nothing below can run."
    )
if len(anchors) < 4:
    print(f"\nOnly {len(anchors)} satellite(s) are usable. A position fix needs four,")
    print("so section 8 will skip; section 9 holds the position and solves the clock,")
    print("which needs only one.")

### The ephemeris the satellites themselves sent

Where a message type 10, an 11 and a 30-series clock message were all decoded from
the same satellite, the CNAV ephemeris can be assembled and compared against
`brdc`. The comparison is in predicted position, not in parameters — the two
messages describe the same orbit with different algebra.

A satellite that produced only one or two of the three gets `None`; the
`toe`/`toc` agreement check refuses to mix halves of different data sets.

In [ ]:
decoded_ephemerides = {}
for sid, result in decodes.items():
    if not result.synced:
        continue
    eph = cnav.assemble_ephemeris(result, sat_id=sid)
    if eph is None:
        types = sorted({m.message_type for m in result.messages})
        print(f"{sid}: cannot assemble a CNAV ephemeris from types {types} "
              "(needs 10, 11 and one of 30-37, all sharing a reference time)")
        continue
    decoded_ephemerides[sid] = eph

    if sid in broadcast:
        t = anchors[sid].tow_s
        from_cnav = eph.orbit_state(t).position_ecef_m
        from_brdc = broadcast[sid].orbit_state(t).position_ecef_m
        print(f"{sid}: decoded CNAV vs downloaded LNAV position differ by "
              f"{np.linalg.norm(from_cnav - from_brdc):8.2f} m")
    else:
        print(f"{sid}: decoded a CNAV ephemeris (nothing to compare it against)")

if not decoded_ephemerides:
    print("\nNo complete CNAV ephemeris this run. Types 10, 11 and a 30-series clock")
    print("message are not adjacent in the broadcast sequence, so a short capture")
    print("often catches only one or two of the three. The fix below uses the")
    print("downloaded broadcast ephemeris either way.")

## 6. Skyplot

Where the satellites were, from the antenna's point of view.

Two things to read off it. The first is geometry: fixes are precise in the
directions satellites are spread across and imprecise where they are not, which is
what DOP puts a number on later. The second is what the front end managed — every
satellite above the horizon is drawn, and the ones that were actually acquired are
highlighted and coloured by their tracked C/N0. The ones that were missed are
almost always the low ones.

In [ ]:
# Every GPS satellite with a usable ephemeris, whether this receiver saw it or not.
all_visible = broadcast_ephemeris.ephemerides_for(
    [f"G{prn:02d}" for prn in range(1, 33)],
    reference_tow, week, epoch_datetime, records=records, verbose=False,
)

# A track, not a point: a GPS satellite moves a few degrees in a minute, which is
# visible over this collect and worth drawing.
track_times = reference_tow + np.linspace(0.0, run.shortest_duration_ms * 1e-3, 12)
azimuth, elevation = {}, {}
for sid, eph in all_visible.items():
    positions = np.array([eph.orbit_state(t).position_ecef_m for t in track_times])
    az, el = navigation.sky_positions(reference_ecef, positions)
    if el.max() > 0:
        azimuth[sid], elevation[sid] = az, el

# Mean C/N0 over the tracked span, on the data component.
cn0 = {}
for sid in run.signal_ids:
    values = run[sid].outputs.cn0_dbhz[run[sid].outputs.cn0_valid]
    if len(values):
        cn0[sid] = float(np.nanmean(values[:, 0]))

fig = plt.figure(figsize=(8, 8), dpi=150)
utils.plotting.plot_skyplot(
    fig, azimuth, elevation,
    cn0_dbhz=cn0,
    tracked=run.signal_ids,
    title=f"{collect_id}  ({len(azimuth)} above the horizon, {len(run)} tracked)",
)
plt.show()

print("Tracked satellites:")
for sid in sorted(run.signal_ids):
    if sid in elevation:
        print(f"  {sid}: elevation {elevation[sid].mean():5.1f} deg, "
              f"azimuth {azimuth[sid].mean():6.1f} deg, "
              f"C/N0 {cn0.get(sid, float('nan')):.1f} dB-Hz")
    else:
        print(f"  {sid}: tracked, but no ephemeris to place it in the sky")

## 7. Pseudoranges

### Forming them

Each channel's epochs land on its own code-phase grid, so no two satellites produce
a measurement at the same instant. Every channel's code phase is therefore
interpolated onto one common receiver-clock grid — linear interpolation, which is
exact to well under a millimetre over a 10 ms epoch spacing.

The nominal receive time is arbitrary. Its offset from GPS time is precisely the
receiver clock bias, and that is what section 9 solves for.

### What the corrections are worth

The magnitudes span five orders of magnitude, which is the single most useful thing
to see here:

| Correction | Size | Why |
|---|---|---|
| **Satellite clock** | tens of km | the SV's clock is free-running; the polynomial and relativity together are ~100 µs |
| **Sagnac** | tens of m | the earth turns ~30 m of arc during a 70 ms transit |
| **Ionosphere** | metres to tens of m | dispersive, so worse on L5 than L1 by (f₁/f₅)² ≈ 1.79 |
| **Troposphere** | a few m at zenith | not dispersive — no frequency pair removes it |
| **Group delay** | ~ns, so a metre or less | T_GD plus the L5 inter-signal correction |

An uncorrected fix is not merely inaccurate; it does not converge to anywhere near
the right place.

In [ ]:
obs = observables.form_observables(
    {sid: run[sid].outputs for sid in anchors},
    anchors,
    epoch_interval_ms=EPOCH_INTERVAL_MS,
    week=week,
)
print(f"{obs.num_epochs:,} measurement epochs over "
      f"{(obs.epoch_uptime_ms[-1] - obs.epoch_uptime_ms[0]) / 1000:.1f} s")
print(f"Satellites: {', '.join(obs.sat_ids)}")

counts = obs.satellites_per_epoch()
print(f"Satellites per epoch: min {counts.min()}, max {counts.max()}")
if counts.min() < 4:
    print(f"  {int((counts < 4).sum())} epoch(s) have fewer than four and cannot be solved.")

raw = obs.pseudorange_m
print(f"\nRaw pseudoranges: {np.nanmin(raw) / 1e3:,.1f} to "
      f"{np.nanmax(raw) / 1e3:,.1f} km")
print("  (a GPS satellite is 20,000-26,000 km away; the offset from that is the")
print("   receiver clock bias, which is arbitrary and about to be solved for)")

In [ ]:
# Klobuchar coefficients, preferably from a CNAV message this receiver decoded.
iono_alpha = iono_beta = None
for sid, result in decodes.items():
    clocks = [m for m in result.messages if 30 <= m.message_type <= 37]
    if clocks:
        parsed = cnav.parse_type_30(clocks[0])
        iono_alpha, iono_beta = parsed.alpha, parsed.beta
        print(f"Klobuchar coefficients from {sid}, message type {clocks[0].message_type}")
        break

if iono_alpha is None:
    # Fall back to the same eight numbers out of the broadcast file's header.  They
    # are updated at most every few days, so the file's copy is the same model the
    # satellites were transmitting -- it just did not happen to arrive in the few
    # seconds this receiver was listening.
    header_iono = broadcast_ephemeris.load_brdc_iono(collect_date)
    if header_iono is not None:
        iono_alpha, iono_beta = header_iono
        print("No CNAV type 30 decoded; using the coefficients from the brdc header.")

if iono_alpha is None:
    print("No ionospheric model available. On single-frequency L5 that leaves")
    print("several metres of range error, mostly in the vertical.")
else:
    print(f"  alpha {iono_alpha}")
    print(f"  beta  {iono_beta}")

corrected, terms = navigation.correct_pseudoranges(
    obs, broadcast,
    signal_type_id=SIGNAL_ID.signal_type_id,
    settings=CORRECTIONS,
    receiver_position_ecef_m=reference_ecef,
    iono_alpha=iono_alpha,
    iono_beta=iono_beta,
)

fig = plt.figure(figsize=(9, 4), dpi=150)
utils.plotting.plot_correction_magnitudes(fig, terms, obs.sat_ids)
plt.show()

## 8. Position solution

Four unknowns — three of position, one of clock — from four or more pseudoranges,
by Newton's method on the geometry.

**DOP** is how geometry turns measurement error into position error: a fix is
precise in the directions the satellites are spread across. VDOP always exceeds
HDOP for a ground receiver, because every satellite is above the antenna and none
below, which is why vertical scatter is the largest of the three.

**Residuals** are the diagnostic. Read their structure, not just their size:
scatter about zero is measurement noise; one satellite consistently off is an
ephemeris or multipath problem on that satellite; all of them drifting together
means something common-mode is unmodelled.

> **If fewer than four satellites survived to here**, no position fix exists — four
> unknowns need four equations, and an under-determined "fix" is not a degraded
> answer but a meaningless one. The next cell says so and moves on; section 9 still
> produces a clock solution, because holding the position fixed removes three of
> the four unknowns. That is not a workaround — it is exactly how a timing receiver
> at a surveyed site operates.

In [ ]:
can_fix_position = len(obs.sat_ids) >= 4

if not can_fix_position:
    print(f"Only {len(obs.sat_ids)} satellite(s) with both a decoded time and an")
    print("ephemeris: G" + ", G".join(s[1:] for s in obs.sat_ids))
    print()
    print("Four are needed for a position fix. Skipping to section 9, which holds the")
    print("position at the reference and solves the clock alone -- one satellite is")
    print("enough for that, and the residuals are more informative than a fix's would")
    print("have been.")
    solution = None
else:
    solution = navigation.solve_series(
        obs, broadcast,
        signal_type_id=SIGNAL_ID.signal_type_id,
        settings=CORRECTIONS,
        initial_position_ecef_m=reference_ecef,
        iono_alpha=iono_alpha,
        iono_beta=iono_beta,
    )

    good = solution.valid
    print(f"{good.sum():,} of {len(good):,} epochs solved")

    mean_ecef = solution.position_ecef_m[good].mean(axis=0)
    lat, lon, height = navigation.ecef_to_geodetic(mean_ecef)
    print(f"\nMean fix:  {lat:.6f} deg N, {lon:.6f} deg E, {height:,.1f} m")
    print(f"Reference: {RECEIVER_REFERENCE_GEO[0]:.6f} deg N, "
          f"{RECEIVER_REFERENCE_GEO[1]:.6f} deg E, {RECEIVER_REFERENCE_GEO[2]:,.1f} m")
    print(f"Offset:    {np.linalg.norm(mean_ecef - reference_ecef):,.1f} m")
    print("  (the reference is a coarse value, so this is a sanity check on the")
    print("   chain, not a measurement of accuracy)")

    for name in ("gdop", "pdop", "hdop", "vdop", "tdop"):
        print(f"  {name.upper():>5}: {np.nanmean(solution.dop[name][good]):5.2f}")

    if len(obs.sat_ids) == 4:
        print("\nExactly four satellites: the fix is exactly determined, so the")
        print("post-fit residuals are zero by construction and carry no information.")

In [ ]:
if solution is None:
    print("No position solution to plot -- see the cell above.")
else:
    enu = solution.enu_about(reference_ecef)
    position_time_s = (solution.epoch_uptime_ms - solution.epoch_uptime_ms[0]) * 1e-3
    fig = plt.figure(figsize=(12, 5), dpi=150)
    utils.plotting.plot_position_enu(
        fig, enu, position_time_s, title=f"Position: {collect_id}"
    )
    plt.show()

In [ ]:
if solution is None:
    print("No position solution, so no post-fit residuals -- section 9 produces the")
    print("more informative ones anyway, against a held position.")
else:
    fig = plt.figure(figsize=(11, 4), dpi=150)
    utils.plotting.plot_pseudorange_residuals(
        fig, position_time_s, solution.residuals_m, obs.sat_ids
    )
    plt.show()

## 9. Clock solution

The receiver's clock is the sample counter, so this is a measurement of the front
end's oscillator.

The **slope** is the rate error in parts per million. A free-running TCXO of the
kind in a USRP is typically within a few ppm; an OCXO would be far better. The
**residual** about that line is what a constant rate does not explain —
measurement noise, plus any real instability in the oscillator.

The position is **held** at the reference here rather than solved alongside. Two
reasons, and the second matters even when a full fix was available:

- it needs only one satellite, so a clock solution exists whatever the satellite
  count;
- with the position fixed, the residuals are no longer forced to zero by the
  geometry. Four satellites against four unknowns leave nothing to see; four
  satellites against a known position leave three degrees of freedom, and a real
  measurement error shows up instead of being absorbed into the fix.

### The independent cross-check, and why its sign flips

One reference oscillator drives both the sampling clock and the downconverting
LO, so a rate error δ shows up twice — and with **opposite** signs.

- **In time.** A fast sample clock makes receiver time run ahead of GPS time, so
  the clock bias grows: slope `+δ·c`.
- **In frequency.** The LO is also high, at `f_nom·(1 + δ)`, so everything
  downconverts too low: every satellite's Doppler is offset by `−δ·f_carrier`.

On L5, one ppm is 1176 Hz — so a couple of ppm is a couple of kilohertz of
common-mode Doppler, which is enormous next to real satellite Doppler of a few
kHz and is why it cannot be ignored. The two numbers below should have the same
magnitude and opposite signs. That they do is a check on the whole chain, because
they arrive by completely different routes: one from decoded time and ephemerides,
the other from the carrier tracking loops.

In [ ]:
clock = navigation.solve_clock_series(
    obs, broadcast, reference_ecef,
    signal_type_id=SIGNAL_ID.signal_type_id,
    settings=CORRECTIONS,
    iono_alpha=iono_alpha,
    iono_beta=iono_beta,
)
time_s = (clock.epoch_uptime_ms - clock.epoch_uptime_ms[0]) * 1e-3

fig = plt.figure(figsize=(11, 6), dpi=150)
utils.plotting.plot_clock_solution(
    fig, time_s, clock.clock_bias_m,
    title=f"Receiver clock, position held: {collect_id}",
)
plt.show()

drift_ppm, rms_m = clock.clock_drift_ppm()
hz_per_ppm = SIGNAL_ID.carrier_freq_hz * 1e-6
print(f"Clock drift: {drift_ppm:+.3f} ppm     residual RMS about the line: {rms_m:.2f} m")
print(f"One ppm on {SIGNAL_ID.link.id.value} is {hz_per_ppm:,.0f} Hz, so this predicts a")
print(f"common-mode Doppler offset of {-drift_ppm * hz_per_ppm:+,.0f} Hz "
      "-- note the sign flip.")

### Residuals against the held position

These are the honest measurement diagnostic. Each satellite's deviation from the
common clock offset — so an ephemeris error, a multipath fade, or a low-elevation
troposphere mismodelling shows up on that satellite alone.

Note this includes any error in `RECEIVER_REFERENCE_GEO` itself, projected onto
each line of sight. With a coarse reference position that projection is the
dominant term, and it appears as a *static* per-satellite offset rather than as
scatter.

In [ ]:
fig = plt.figure(figsize=(11, 4), dpi=150)
utils.plotting.plot_pseudorange_residuals(
    fig, time_s, clock.residuals_m, obs.sat_ids,
    title="Residuals about the held position",
)
plt.show()

print("Per-satellite mean residual (static offset) and scatter:")
for j, sid in enumerate(obs.sat_ids):
    column = clock.residuals_m[:, j]
    column = column[np.isfinite(column)]
    if len(column):
        print(f"  {sid}: mean {column.mean():+8.2f} m   std {column.std():6.2f} m")

In [ ]:
# Cross-check: the oscillator rate error should also show up as a common-mode
# offset between the Doppler tracking measured and the Doppler the satellite's own
# motion accounts for.
predicted, measured = [], []
for sid in obs.sat_ids:
    outputs = run[sid].outputs
    locked = outputs.pll_mode[outputs.valid]
    doppler = outputs.doppler_freq_hz[outputs.valid]
    measured.append(float(np.mean(doppler[locked])) if locked.any() else np.nan)

    eph, dt, t = broadcast[sid], 1.0, anchors[sid].tow_s
    before = eph.orbit_state(t - dt / 2).position_ecef_m
    after = eph.orbit_state(t + dt / 2).position_ecef_m
    line_of_sight = (before + after) / 2 - reference_ecef
    line_of_sight /= np.linalg.norm(line_of_sight)
    range_rate = np.dot((after - before) / dt, line_of_sight)
    predicted.append(-range_rate / 2.99792458e8 * SIGNAL_ID.carrier_freq_hz)

offsets = np.array(measured) - np.array(predicted)
print("Measured minus predicted Doppler:")
for sid, offset in zip(obs.sat_ids, offsets):
    print(f"  {sid}: {offset:+8.1f} Hz")
common = np.nanmean(offsets)
print(f"  common-mode: {common:+.1f} Hz  ->  "
      f"{common / SIGNAL_ID.carrier_freq_hz * 1e6:+.3f} ppm")
implied_ppm = common / SIGNAL_ID.carrier_freq_hz * 1e6
print(f"\nClock solution said   {drift_ppm:+.3f} ppm")
print(f"Doppler implies       {implied_ppm:+.3f} ppm")
print(f"Same magnitude, opposite sign, agreeing to "
      f"{abs(abs(implied_ppm) - abs(drift_ppm)):.3f} ppm.")
print("The spread across satellites is ephemeris error plus the reference position")
print("being approximate; the common-mode part is the oscillator.")

## 10. Broadcast versus precise orbits

How much of the error arrived with the signal rather than being made by the
receiver.

IGS precise orbits are computed after the fact from a global station network and
are good to a few centimetres. The broadcast ephemeris is a prediction, and is
good to a metre or two. That difference is a floor on any fix: no amount of care
in the receiver removes it.

`utils.precise_orbits` fetches the file, for the same reason
`utils.broadcast_ephemeris` does — `gnss_tools.orbits.sp3_utils` downloads with no
credentials at all, and CDDIS has required an Earthdata login for years. Its
parsing and spline interpolation are used unchanged.

In [ ]:
if not USE_PRECISE_ORBIT_CHECK:
    print("Skipped (USE_PRECISE_ORBIT_CHECK is False).")
else:
    try:
        from gnss_tools.orbits.sp3_utils import (
            compute_splines_from_sp3_dict,
            compute_values_from_sp3_splines,
            download_and_parse_sp3_data,
        )
        from utils import precise_orbits

        # Fetch it ourselves, into the directory sp3_utils caches in, so that its
        # own (credential-less, and therefore broken) download never runs.
        precise_orbits.download_sp3(collect_date)

        resources = str(utils.environment_variables.get_resources_path())
        gps_seconds = week * 604800.0 + reference_tow
        sp3 = download_and_parse_sp3_data(
            gps_seconds - 3600.0, gps_seconds + 3600.0, resources
        )
        print(f"SP3: {len(sp3.epochs)} epochs, {len(sp3.position)} satellites "
              "(all constellations)")

        splines = compute_splines_from_sp3_dict(sp3.epochs, sp3.position)
        precise = compute_values_from_sp3_splines(np.array([gps_seconds]), splines)

        print("\nBroadcast minus precise, at the decoded epoch:")
        errors = []
        for sid in sorted(broadcast):
            if sid not in precise:
                print(f"  {sid}: not in the SP3 file")
                continue
            # Positions come back in metres, not the kilometres the SP3 text file
            # stores -- the parser has already scaled them.
            truth = np.asarray(precise[sid]).ravel()[:3]
            predicted = broadcast[sid].orbit_state(reference_tow).position_ecef_m
            error = float(np.linalg.norm(predicted - truth))
            errors.append(error)
            print(f"  {sid}: {error:6.2f} m")

        if errors:
            print(f"\nMean {np.mean(errors):.2f} m. A metre or two is normal, and it is")
            print("error the signal arrived with rather than error the receiver made.")
            print("It sets a floor on the fix above that no receiver-side work removes.")
    except Exception as exc:
        print(f"Precise orbit check unavailable: {type(exc).__name__}: {exc}")
        print("It needs a separate CDDIS download; nothing above depends on it.")

---

## Exercises

1. **Turn a correction off.** Set `satellite_clock=False` in `CORRECTIONS` and re-run
   from section 7. Predict first: does the fix move by metres, kilometres, or fail
   to converge? Then try `sagnac=False`, which is a much smaller change and moves
   the answer in a specific direction — which one, and why?

2. **Measurement rate.** Set `EPOCH_INTERVAL_MS` to 1000. The scatter should not
   change much but the clock drift estimate should get *worse*. Why does fewer
   points hurt a slope more than it hurts a mean?

3. **Drop a satellite.** Remove the highest-elevation satellite from `anchors` before
   section 7 and re-run. Compare HDOP and VDOP before and after. Now drop the
   lowest instead. Which costs more, and does DOP predict it?

4. **The ionosphere on L5.** Set `ionosphere=False`. The change is larger than the
   equivalent on L1 by (f₁/f₅)² ≈ 1.79 — check that the shift is in the direction
   you expect, and mostly in the vertical.

5. **Where the residuals come from.** Section 8's residuals include everything
   unmodelled. Estimate how much is broadcast ephemeris error using section 10's
   numbers, and see how much is left over.

6. **A shorter decode.** Re-run notebook `01` with `TRACK_DURATION_MS = 12000` and
   come back. Does the decode still find a message? Does it find two? What breaks
   first — the time anchor, or the ephemeris assembly?

## Not covered here

**Carrier-phase positioning.** Tracking measures carrier phase to a fraction of a
cycle — millimetres of range — but with an unknown integer number of cycles.
Resolving those integers is what takes a receiver from metres to centimetres, and
it is a much larger subject than the code-phase solution here.

**RAIM and fault detection.** With five satellites and four unknowns there is one
degree of redundancy, which is enough to notice a bad measurement but not to
identify which one. Real receivers need considerably more.

**Filtering.** Every epoch here is solved independently, which is why the scatter
looks like scatter. A Kalman filter carrying position and clock between epochs
would use the fact that neither changes much in 100 ms.

**Multi-constellation and multi-frequency.** The single largest error left in this
fix is the ionosphere, and the real answer to it is a second frequency — which the
L5-only collects here cannot provide.

**CNAV-2's ephemeris.** L1C's subframe 1 decodes fully, so its *time* is available,
but subframes 2 and 3 sit behind LDPC codes that are not implemented yet. See
`utils/nav/cnav2.py`.